In [ ]:
# Gemma re-ranking for "Who's the Best Pitcher"
#
# The deterministic solver already puts the right *region* of the JSON in its
# shortlist almost every time -- the structural audit reads 92-100% across every
# question pattern -- yet it scores 0.641. That gap is "right region, wrong
# leaf": neighbouring leaves in the same group, where `outs.ktotal` is batters
# struck out and `outcome.ktotal` is strikes thrown, four times larger.
#
# So the model is not asked to answer the question. It is asked to CHOOSE among
# eight candidates the solver already found, each rendered in English with its
# group spelled out. The chosen candidate's value is then copied verbatim from
# the source.
#
# The model never writes a number. Under exact-match scoring that is the whole
# point: a generated "123.45" loses to a stored "123.4500", and a small model
# rewrites numbers constantly.
#
# An earlier attempt scored 0.478, BELOW the plain heuristic, by showing Gemma
# raw paths like `statistics.pitching.overall.outs.ktotal` and asking for an
# index. The difference here is the rendering, not the model.
import json, re, time
import pandas as pd, torch

# The dataset directory is named after whatever title the upload was given, and
# the file may sit one level down, so find it instead of assuming the slug.
import glob
hits = sorted(glob.glob("/kaggle/input/**/candidates.json", recursive=True))
if not hits:
    print("candidates.json not found. Everything currently attached:")
    for p in sorted(glob.glob("/kaggle/input/*/**", recursive=True))[:60]:
        print("  ", p)
    raise SystemExit("Attach candidates.json as a Dataset, then re-run this cell.")
CAND = hits[0]
print("using", CAND)
rows = json.load(open(CAND))

# ---- CANDIDATE SANITISER ----------------------------------------------------
# Measured on the shipped 0.68478 answers, 7 are structurally impossible:
#   4x a statistical question answered from `game_number` (a game's sequence
#     number, not a statistic -- the exact signature of the `ip_1` bug, where a
#     leaf's NAME matches loosely but its SEMANTICS are wrong)
#   2x a type error: "WHIP" -> 'A', "at-bats" -> 'Toronto'
#   1x a raw UUID
#
# Dropping them costs nothing when the solver was right anyway, and converts a
# guaranteed-wrong answer into a plausible one when it was not. It also shortens
# the list the model must choose from, which is the one thing measured to help:
# the shortlist holds ~5.4 distinct values out of 8, so a third of the decision
# is redundant.
import re as _re
_UUID = _re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', _re.I)
_NUMQ = _re.compile(r'\b(how many|how much|what (?:was|is) the (?:number|total|average|'
                    r'percentage|ratio|count)|era|whip|ops|obp|slugging|batting average|'
                    r'innings|pitches|strikeouts|rbis?|at-bats|putouts|errors|walks)\b', _re.I)
_BADLEAF = ('game_number', "'id'", '"id"', 'uuid', 'reference')

def _numeric(v):
    try: float(str(v).strip().lstrip('.').replace(',', '')); return True
    except Exception: return False

def _ok(c, q):
    v = str(c.get('value', '')).strip()
    if not v or _UUID.match(v): return False
    pth = str(c.get('path', '')).lower()
    if any(b in pth for b in _BADLEAF): return False
    if _NUMQ.search(q) and not _numeric(v): return False
    return True

_dropped = _emptied = 0
for _r in rows:
    _cs = _r.get('candidates') or []
    _keep = [c for c in _cs if _ok(c, _r['question'])]
    if not _keep:                      # never leave a question with nothing
        _keep = _cs; _emptied += 1
    _dropped += len(_cs) - len(_keep)
    _r['candidates'] = _keep
print(f'sanitiser: dropped {_dropped} candidates; '
      f'{_emptied} questions kept their originals (all candidates would have been filtered)')
print('top-1 changed by sanitising alone:',
      sum(1 for _r, _o in zip(rows, json.load(open(CAND)))
          if _r['candidates'] and _o['candidates']
          and _r['candidates'][0]['value'] != _o['candidates'][0]['value']))
print(f"{len(rows)} questions, {sum(len(r['candidates']) for r in rows)} candidates")
print(rows[0]["question"])
for i, c in enumerate(rows[0]["candidates"][:3], 1):
    print(f"  [{i}] {c['text'][:120]}")


In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import glob, os

# Find the attached Gemma rather than hardcoding a path: the directory depends
# on which variant was added, and more than one may be attached.
hits = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
        if os.path.exists(os.path.join(d, "config.json"))]
print("model dirs found:")
for h in hits:
    print("  ", h)
# Prefer a 27B instruction-tuned checkpoint; fall back to whatever is attached.
pref = [h for h in hits if "27b" in h.lower()] or \
       [h for h in hits if "4b" in h.lower() and "it" in h.lower()] or hits
MODEL = pref[0].rstrip("/")
print("using:", MODEL)
# Kaggle meters GPU/TPU but not CPU, so this runs with zero GPU quota -- just
# slower. On GPU: float16, because T4 and P100 are pre-Ampere and emulate bf16.
# On CPU: float32, because CPU bf16 matmul without AMX is slower than fp32.
CUDA = torch.cuda.is_available()
if CUDA:
    # is_bf16_supported() counts EMULATED support, so it returns True on a T4
    # (Turing, SM 7.5) which has no native bf16 -- torch then emulates it:
    # slower, and 8 mantissa bits instead of fp16's 11 feeding the digit
    # softmax. Gate on compute capability instead: Ampere (SM 8.0) and up.
    _cc = torch.cuda.get_device_capability()[0]
    DTYPE = torch.bfloat16 if _cc >= 8 else torch.float16
    print(f'compute capability {torch.cuda.get_device_capability()} -> {DTYPE}')
else:
    DTYPE = torch.float32
    torch.set_num_threads(os.cpu_count())
# A 27B will not fit a T4 in fp16 (54 GB of weights against 16 GB), so load it
# 4-bit NF4 -- about 14 GB, which fits a single T4 with room for activations.
# Anything smaller loads normally.
BIG = "27b" in MODEL.lower()
if BIG and CUDA:
    from transformers import BitsAndBytesConfig
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_use_double_quant=True)
    print("loading 27B in 4-bit NF4 ...", flush=True)
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL, device_map="auto", quantization_config=qcfg).eval()
else:
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL, device_map="auto" if CUDA else None, torch_dtype=DTYPE).eval()
print(f"device={'cuda' if CUDA else 'cpu'} dtype={DTYPE} threads={os.cpu_count()}")
processor = AutoProcessor.from_pretrained(MODEL)
print("loaded", torch.cuda.get_device_name(0) if CUDA else "on CPU")


In [ ]:
import numpy as np
import zlib
SYSTEM = (
    "You match baseball questions to the correct field of a statistics database. "
    "You are given a question and a numbered list of candidate fields, each shown "
    "with the group it belongs to and its value. Reply with ONLY the number of the "
    "candidate that answers the question. No words, no explanation."
)

# Read the answer off the LOGITS rather than the generated text.
#
# Generating an index and regexing it out throws away everything except the
# argmax, and a malformed reply is scored as "no opinion". One forward pass
# already contains a full distribution over the digit tokens, which costs the
# same, cannot fail to parse, and yields a CONFIDENCE -- so the model can be
# allowed to override the solver only when it is actually sure.
DIGIT_IDS = None

def digit_ids(n):
    global DIGIT_IDS
    if DIGIT_IDS is None:
        tok = processor.tokenizer
        DIGIT_IDS = []
        for d in range(1, 10):
            ids = tok.encode(str(d), add_special_tokens=False)
            DIGIT_IDS.append(ids[0] if ids else -1)
    return DIGIT_IDS[:n]

# PERMUTATION VOTING.
#
# Candidate [1] is ALWAYS the solver's top pick, and an LLM asked to choose a
# number favours the first option. That alone explains the measured fact that
# every scored submission from v4 on is byte-identical to the solver's own top-1:
# the model never moved a single answer. A confidence floor cannot fix this,
# because the bias is in the PICK, not in the certainty.
#
# So ask the same question with the candidates SHUFFLED, map each answer back to
# its original index, and take the majority. Position bias averages out, and the
# vote share becomes a real confidence -- unlike a single softmax over digits,
# which is confidently wrong exactly when the bias is strongest.
# On CPU a forward pass is ~60 s, so 3 orderings is all that fits (~4.5 h for 87
# gated questions). On GPU it is ~1 s, so buy finer vote resolution: 7 orderings
# gives vote shares in sevenths instead of thirds, which makes the confidence
# gate meaningful, and still costs only minutes.
# A 27B in 4-bit runs roughly 5-10x slower per call than the 4B. 5 orderings
# still de-biases position and gives gate resolution in fifths, which is what
# matters -- the 4B result was monotone in the gate (unanimous 63/92, 2/3 62,
# 1/3 60), so the gate must stay TIGHT and needs to be expressible.
PERMS = 5 if CUDA else 3

def _ask(question, cands):
    """(1-based index into `cands`, probability) for this exact ordering."""
    lines = [f"[{i}] {c['text']}" for i, c in enumerate(cands, 1)]
    user = ("Question: " + question + "\n\nCandidate fields:\n"
            + "\n".join(lines) + "\n\nWhich number answers the question?")
    messages = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
                {"role": "user", "content": [{"type": "text", "text": user}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        logits = model(**inputs).logits[0, -1]
    ids = digit_ids(len(cands))
    sel = torch.tensor([i for i in ids if i >= 0], device=logits.device)
    pr = torch.softmax(logits[sel].float(), dim=0)
    k = int(pr.argmax())
    return k + 1, float(pr[k])

def choose(row):
    """(index, vote share) after voting over PERMS orderings."""
    cands = row["candidates"]; n = len(cands)
    rng = np.random.default_rng(zlib.crc32(str(row["ID"]).encode()))  # crc32, NOT hash(): Python randomises str hashing per process (PYTHONHASHSEED),
    # so hash() gave a DIFFERENT permutation order every run -- the notebook could not
    # reproduce its own submission, which is exactly what the deliverable rule forbids.
    votes = np.zeros(n + 1)
    for r in range(PERMS):
        order = np.arange(n) if r == 0 else rng.permutation(n)   # keep one identity pass
        k_shuf, _ = _ask(row["question"], [cands[i] for i in order])
        k_shuf = min(max(k_shuf, 1), n)
        votes[order[k_shuf - 1] + 1] += 1
    k = int(votes.argmax())
    return k, float(votes[k] / PERMS)


In [ ]:
# Only ask the model where the solver is genuinely unsure.
#
# The margin is the score gap between the top candidate and the best candidate
# holding a DIFFERENT value -- a gap of zero means the solver is breaking a tie
# arbitrarily, which is where a re-ranker has something to add. Where the gap is
# wide the solver has a real reason for its pick, and overriding it is how the
# previous attempt scored 0.478.
#
# Run 1 used MARGIN=2.0 (90 questions) and scored 61/92. Run 2 widened this to
# 8.0 (163 questions) and REGRESSED to 60/92, so the gate is back at 2.0.
# Widening has now lost every time it was tried on this competition: questions
# just outside the tightest band are ones where the solver had a real reason for
# its pick, and overriding those costs more than the extra coverage earns.
MARGIN = 2.0

def margin(row):
    c = row["candidates"]
    if len(c) < 2:
        return 99.0
    top = c[0]["value"]
    for x in c[1:]:
        if x["value"] != top:
            return c[0]["score"] - x["score"]
    return 99.0            # every candidate agrees; nothing to choose

eligible = [i for i, r in enumerate(rows) if r["candidates"] and margin(r) < MARGIN]
# tightest margins first, so a run that has to be cut short has still spent its
# time on the questions where the solver was closest to a coin flip
eligible.sort(key=lambda i: margin(rows[i]))
MAX_Q = len(eligible) if CUDA else min(len(eligible), 200)
eligible = eligible[:MAX_Q]
print(f"{len(eligible)}/{len(rows)} questions to re-rank "
      f"(margin < {MARGIN}, {'GPU' if CUDA else 'CPU, capped'})")

# Only override the solver when the model is actually confident. Its first run
# gained +2 questions ungated; a confidence floor lets the gate be widened to
# many more questions without paying for the low-conviction picks.
# Was 0.50. Every submission from v4 on came out byte-identical to the solver's
# own top-1, so the model moved NOTHING -- and since k=0 and k=1 both resolve to
# cands[0], only k>=2 can change an answer. 0.25 turns the re-ranker on; the
# sweep in the next cell writes a file at every gate so this choice is revisable
# without paying for the model again.
# MIN_P is now a VOTE SHARE, not a softmax probability. With PERMS=3 the only
# reachable values are 1/3, 2/3 and 1, so 0.66 means "at least 2 of 3 orderings
# agreed" -- a far more meaningful gate than a single digit logit.
MIN_P = 0.8 if PERMS == 5 else 0.66 if PERMS == 3 else 4/7

import numpy as np

# ---- incremental re-rank ---------------------------------------------------
# A solver fix typically changes 2-5 shortlists, but a full pass re-ranks every
# gated question at ~60 s each -- 90 minutes to test a one-line change, which
# makes "one change per submission" unaffordable and pushes you into batching
# fixes you then cannot attribute.
#
# So cache each question's shortlist SIGNATURE (its candidate paths, in order)
# together with the pick it produced. On the next run a question only needs the
# model again if its shortlist actually moved. Keying on the signature rather
# than the question ID is what makes this safe: any change to the candidates --
# different paths, different order -- misses the cache and gets re-asked.
def sig(row):
    # The signature must cover the PROMPT, not just the paths. A change to how
    # candidates are RENDERED (naming the home/away clubs, say) leaves every
    # path identical while changing what the model is asked -- a path-only key
    # would reuse all 87 stale picks and silently skip the model entirely.
    return [str(PERMS)] + [c["text"] for c in row["candidates"]]

CACHE = {}
_hits = sorted(glob.glob("/kaggle/input/**/rerank_cache.json", recursive=True))
if _hits:
    CACHE = json.load(open(_hits[0]))
    print(f"re-rank cache: {len(CACHE)} entries from {_hits[0]}")
else:
    print("no rerank_cache.json attached -- full pass this run.")
    print("attach the cache this run writes as a Dataset next time to re-rank only what changed.")

picks, confs = [0] * len(rows), []
todo, reused = [], 0
for i in eligible:
    hit = CACHE.get(str(rows[i]["ID"]))
    if hit and hit.get("sig") == sig(rows[i]):
        picks[i] = hit["pick"]
        reused += 1
    else:
        todo.append(i)
print(f"{reused} reused from cache, {len(todo)} to re-rank "
      f"(~{len(todo) * 60 / 60:.0f} min at 60 s/question)")

t0 = time.time()
raw = {}          # i -> (k, p) BEFORE the floor, so gates can be re-cut for free
for n, i in enumerate(todo):
    k, p = choose(rows[i])
    raw[i] = (int(k), float(p))
    picks[i] = k if p >= MIN_P else 0
    if picks[i] > 1:
        confs.append(p)
    if (n + 1) % 20 == 0:
        r = (n + 1) / (time.time() - t0)
        print(f"  {n+1}/{len(todo)}  {r:.2f} q/s  ETA {(len(todo)-n-1)/r/60:.1f} min",
              flush=True)
print(f"done in {(time.time()-t0)/60:.1f} min")
# Separate the three outcomes -- conflating "never asked" with "asked but not
# confident" is what made run 1 impossible to calibrate from its output alone.
agreed   = sum(1 for i in eligible if picks[i] == 1)
rejected = sum(1 for i in eligible if picks[i] == 0)
moved    = sum(1 for i in eligible if picks[i] > 1)
print(f"of {len(eligible)} eligible: {moved} moved, {agreed} agreed with the solver, "
      f"{rejected} rejected by the confidence floor (MIN_P={MIN_P})")
print(f"mean confidence on the moves: {np.mean(confs) if confs else float('nan'):.3f}")
if rejected > len(eligible) * 0.6:
    print("WARNING: the floor is rejecting most picks -- lower MIN_P and re-run")


In [ ]:
# emit the stored string verbatim -- never anything the model produced
answers = []
for row, k in zip(rows, picks):
    cands = row["candidates"]
    if not cands:
        answers.append("no answer")
    else:
        answers.append(cands[(k - 1) if k >= 1 else 0]["value"])

sub = pd.DataFrame({"ID": [r["ID"] for r in rows], "ANSWER": answers})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.head())

# ---- gate sweep -------------------------------------------------------------
# Only k>=2 can change an answer, and MIN_P decides whether it survives. Dump the
# raw confidences and emit one file per gate: choosing the gate then costs no
# extra model time, instead of another 90-minute pass per threshold.
raw = globals().get("raw", {})
wanted = {i: kp for i, kp in raw.items() if kp[0] >= 2}
print(f"\nmodel wanted to MOVE {len(wanted)} of {len(raw)} questions it was asked")
if wanted:
    ps = np.array([kp[1] for kp in wanted.values()])
    print("  confidence on those moves: " + "  ".join(
        f"p{q}={v:.3f}" for q, v in zip([10,25,50,75,90], np.percentile(ps,[10,25,50,75,90]))))
    print(f"  survive 0.50: {(ps>=0.50).sum()}   0.30: {(ps>=0.30).sum()}   "
          f"0.25: {(ps>=0.25).sum()}   0.00: {len(ps)}")
else:
    print("  the model never preferred a non-top-1 candidate. MIN_P is NOT the blocker --")
    print("  the chooser is. Candidate 1 is always the solver's pick, so this is position")
    print("  bias; permutation voting, not a lower gate, is the fix.")

# reachable vote shares only -- a gate between two multiples of 1/PERMS is
# identical to the lower one and just wastes a file
GATES = sorted({round(v/PERMS, 4) for v in range(1, PERMS+1)})
# every reachable vote share, not just >=0.5: the re-ranker has never moved a single
# answer in any shipped submission, so the PERMISSIVE gates are where a first gain
# would show. Each extra file is free once the model pass is done.
for gate in GATES:
    gp = list(picks)
    for i, (k, pv) in raw.items():
        gp[i] = k if pv >= gate else 0
    ans = [("no answer" if not r["candidates"]
            else r["candidates"][(k-1) if k >= 1 else 0]["value"])
           for r, k in zip(rows, gp)]
    f = f"/kaggle/working/submission_vote{int(round(gate*100)):03d}.csv"
    pd.DataFrame({"ID": [r["ID"] for r in rows], "ANSWER": ans}).to_csv(f, index=False)
    print(f"  MIN_P={gate:.2f} -> {f.split('/')[-1]}  "
          f"({sum(1 for a,b in zip(ans,answers) if a!=b)} differ from the MIN_P={MIN_P} file)")

# Write the cache for the next run. Only gated questions are stored, because
# ungated ones never cost a model call in the first place.
cache_out = {str(rows[i]["ID"]): {"sig": sig(rows[i]), "pick": picks[i],
                                  "raw": raw.get(i)}
             for i in eligible}
with open("/kaggle/working/rerank_cache.json", "w") as f:
    json.dump(cache_out, f)
print(f"\nwrote rerank_cache.json ({len(cache_out)} entries)")
print("Attach it as a Dataset next run: only questions whose shortlist changed "
      "will be re-ranked.")
